# Cell 1: Setup
Clone repo, install deps, configure paths and device.

In [4]:
# Use Kaggle's pre-installed PyTorch — do NOT reinstall (version mismatch causes CUDA errors)
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')


PyTorch : 2.10.0+cu128
CUDA available: True


In [5]:
!git clone https://github.com/marwa0927/Multimodal-CTR-Prediction.git /kaggle/working/project
%cd /kaggle/working/project
!pip install -q scikit-learn tqdm

import os, sys
sys.path.insert(0, '/kaggle/working/project')

DATA_DIR   = '/kaggle/input/datasets/marwaelazraq/mmctr-data'
OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/features',    exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/submissions', exist_ok=True)

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import time

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

N_GPUS = torch.cuda.device_count()
print(f'GPUs    : {N_GPUS}')
for i in range(N_GPUS):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')

ITEM_EMB_PATH = f'{OUTPUT_DIR}/features/item_emb_task1.npy'
CKPT_T2       = f'{OUTPUT_DIR}/checkpoints/task2_best.pt'
CKPT_T1E      = f'{OUTPUT_DIR}/checkpoints/task1_eval_best.pt'
N_ITEMS       = 91718
MAX_SEQ_LEN   = 64


fatal: destination path '/kaggle/working/project' already exists and is not an empty directory.
/kaggle/working/project
PyTorch : 2.10.0+cu128
CUDA    : True
GPUs    : 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Device  : cuda


# Cell 2: Task 1 — Improved Item Embeddings
TagEncoder with 128-dim tag embeddings + InfoNCE co-occurrence loss.  
Final blend: `0.4 * tag_output + 0.6 * item_emb_d128` (tag-only for zero-vector items).

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# ── Model ─────────────────────────────────────────────────────────────────────
class TagEncoder(nn.Module):
    def __init__(self, vocab_size=11741, tag_dim=128, out_dim=128, max_tags=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, tag_dim, padding_idx=0)
        self.mlp = nn.Sequential(
            nn.Linear(tag_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, out_dim),
        )

    def forward(self, tag_ids):
        mask   = (tag_ids != 0).float().unsqueeze(-1)
        x      = self.emb(tag_ids) * mask
        pooled = x.sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.mlp(pooled)


def infonce_loss(anchors, positives, temperature=0.07):
    a = F.normalize(anchors,   dim=-1)
    p = F.normalize(positives, dim=-1)
    logits = torch.mm(a, p.T) / temperature
    labels = torch.arange(len(a), device=a.device)
    return F.cross_entropy(logits, labels)


# ── Load item_info ─────────────────────────────────────────────────────────────
print('Loading item_info.parquet …')
item_info = pd.read_parquet(f'{DATA_DIR}/item_info.parquet')
print(f'  shape: {item_info.shape}  cols: {item_info.columns.tolist()}')

TAG_VOCAB = 11740
MAX_TAGS  = 64
tag_col = next((c for c in ['item_tags', 'tag_id', 'tags', 'tag_ids'] if c in item_info.columns), None)

print(f'  tag column: {tag_col}')

tags_tensor       = torch.zeros(N_ITEMS + 1, MAX_TAGS, dtype=torch.long)
pretrained_tensor = torch.zeros(N_ITEMS + 1, 128,      dtype=torch.float32)

for _, row in item_info.iterrows():
    iid = int(row['item_id'])
    if not (1 <= iid <= N_ITEMS):
        continue
    if tag_col is not None:
        raw = row[tag_col]
        if isinstance(raw, (list, np.ndarray)):
            t = [int(x) for x in raw if int(x) != 0][:MAX_TAGS]
        elif isinstance(raw, str) and raw.strip():
            t = [int(x) for x in raw.replace(' ', '').split(',') if x][:MAX_TAGS]
        else:
            t = []
        padded = t + [0] * (MAX_TAGS - len(t))
        tags_tensor[iid] = torch.tensor(padded, dtype=torch.long)
    raw_emb = row.get('item_emb_d128', None)
    if raw_emb is not None:
        arr = np.asarray(raw_emb, dtype=np.float32)
        if arr.shape == (128,):
            pretrained_tensor[iid] = torch.tensor(arr)

print(f'  tags_tensor: {tags_tensor.shape}  pretrained: {pretrained_tensor.shape}')

# ── Co-occurrence pairs from item_seq ─────────────────────────────────────────
print('Loading train sequences for co-occurrence …')
train_for_pairs = pd.read_parquet(f'{DATA_DIR}/train.parquet')
seq_col_t1 = next((c for c in ['item_seq', 'item_sequence', 'his_item_id']
                   if c in train_for_pairs.columns), None)
print(f'  seq column: {seq_col_t1}')

pairs = set()
if seq_col_t1 is not None:
    for seq in tqdm(train_for_pairs[seq_col_t1].values[:1_800_000], desc='Pairs'):
        if isinstance(seq, (list, np.ndarray)):
            ids = [int(x) for x in seq if int(x) != 0]
        elif isinstance(seq, str) and seq.strip():
            ids = [int(x) for x in seq.replace(' ', '').split(',') if x and x != '0']
        else:
            continue
        for i in range(len(ids) - 1):
            a, b = ids[i], ids[i+1]
            if 1 <= a <= N_ITEMS and 1 <= b <= N_ITEMS:
                pairs.add((a, b))

pairs = list(pairs)
print(f'  unique pairs: {len(pairs):,}')
del train_for_pairs

# ── Training ───────────────────────────────────────────────────────────────────
DEVICE_T1    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_model_t1    = TagEncoder(vocab_size=TAG_VOCAB + 1).to(DEVICE_T1)
# DataParallel across both T4s
model_t1 = nn.DataParallel(_model_t1) if N_GPUS > 1 else _model_t1
opt_t1   = torch.optim.Adam(_model_t1.parameters(), lr=1e-3)

tags_tensor       = tags_tensor.to(DEVICE_T1)
pretrained_tensor = pretrained_tensor.to(DEVICE_T1)

# Larger batches to saturate both GPUs
TRAIN_BS_T1 = 4096 * N_GPUS
PAIR_BS_T1  = 2048 * N_GPUS

all_ids_t1 = torch.arange(1, N_ITEMS + 1)
loader_t1  = DataLoader(TensorDataset(all_ids_t1), batch_size=TRAIN_BS_T1, shuffle=True)

has_pairs = len(pairs) > 0
if has_pairs:
    pairs_loader = DataLoader(
        TensorDataset(torch.tensor(pairs, dtype=torch.long)),
        batch_size=PAIR_BS_T1, shuffle=True)

EPOCHS_T1 = 50
print(f'\nTraining TagEncoder for {EPOCHS_T1} epochs on {N_GPUS} GPU(s) …')
t0_t1 = time.time()

for epoch in range(1, EPOCHS_T1 + 1):
    model_t1.train()
    total_loss, n_batches = 0.0, 0
    pair_iter = iter(pairs_loader) if has_pairs else None

    for (item_ids,) in loader_t1:
        item_ids = item_ids.to(DEVICE_T1)
        out    = model_t1(tags_tensor[item_ids])
        target = pretrained_tensor[item_ids]
        mse    = F.mse_loss(out, target)

        if pair_iter is not None:
            try:
                (pb,) = next(pair_iter)
            except StopIteration:
                pair_iter = iter(pairs_loader)
                (pb,) = next(pair_iter)
            pb    = pb.to(DEVICE_T1)
            a_out = model_t1(tags_tensor[pb[:, 0]])
            p_out = model_t1(tags_tensor[pb[:, 1]])
            nce   = infonce_loss(a_out, p_out)
        else:
            nce = torch.tensor(0.0, device=DEVICE_T1)

        loss = 0.5 * mse + 0.5 * nce
        opt_t1.zero_grad()
        loss.backward()
        opt_t1.step()
        total_loss += loss.item()
        n_batches  += 1

    if epoch == 1:
        elapsed = time.time() - t0_t1
        print(f'  Epoch 1 done in {elapsed:.1f}s — est. total: {elapsed * EPOCHS_T1 / 60:.1f} min')
    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch {epoch:3d}/{EPOCHS_T1} | Loss: {total_loss / n_batches:.4f}')

# ── Build final embeddings ─────────────────────────────────────────────────────
# Use the underlying module for inference (avoids DataParallel scatter overhead)
_infer_t1 = _model_t1
_infer_t1.eval()
with torch.no_grad():
    _ids     = torch.arange(1, N_ITEMS + 1, device=DEVICE_T1)
    tag_out  = _infer_t1(tags_tensor[_ids]).cpu().numpy()   # (91718, 128)
    pre_np   = pretrained_tensor[_ids].cpu().numpy()        # (91718, 128)

zero_mask = np.abs(pre_np).sum(axis=1) == 0
final_emb = np.where(
    zero_mask[:, None],
    tag_out,
    0.4 * tag_out + 0.6 * pre_np
).astype(np.float32)

np.save(ITEM_EMB_PATH, final_emb)
print(f'\nSaved: {ITEM_EMB_PATH}')
print(f'Shape: {final_emb.shape}')
assert final_emb.shape == (N_ITEMS, 128), f'Shape mismatch: {final_emb.shape}'
print('Shape verified: (91718, 128) \u2713')
print(f'Zero-pretrained items (tag-only blend): {zero_mask.sum():,}')


Loading item_info.parquet …
  shape: (91718, 3)  cols: ['item_id', 'item_tags', 'item_emb_d128']
  tag column: item_tags
  tags_tensor: torch.Size([91719, 64])  pretrained: torch.Size([91719, 128])
Loading train sequences for co-occurrence …
  seq column: item_seq


Pairs: 100%|██████████| 1800000/1800000 [00:29<00:00, 61968.08it/s]


  unique pairs: 5,488,663

Training TagEncoder for 50 epochs on 2 GPU(s) …
  Epoch 1 done in 3.7s — est. total: 3.1 min
  Epoch   1/50 | Loss: 4.5483
  Epoch   5/50 | Loss: 4.2539
  Epoch  10/50 | Loss: 4.2020
  Epoch  15/50 | Loss: 4.1683
  Epoch  20/50 | Loss: 4.1420
  Epoch  25/50 | Loss: 4.1225
  Epoch  30/50 | Loss: 4.1097
  Epoch  35/50 | Loss: 4.0967
  Epoch  40/50 | Loss: 4.0906
  Epoch  45/50 | Loss: 4.0726
  Epoch  50/50 | Loss: 4.0675

Saved: /kaggle/working/outputs/features/item_emb_task1.npy
Shape: (91718, 128)
Shape verified: (91718, 128) ✓
Zero-pretrained items (tag-only blend): 1


# Cell 3: Task 2 — CTR Model Architecture (SASRec + DCNv2 + Target Attention)
One architectural improvement retained: **DIN-style target attention** after the transformer.
- Query = target item repr (192-dim); K/V = all transformer positions (B, L, 192)
- Concat last-position output + attended output → 384-dim sequence representation
- `DCN_IN` 480→672, `COMBINED` 736→928
- Focal loss and aux next-item loss removed: both hurt convergence on this dataset.

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ── Dice activation (DIN paper) ────────────────────────────────────────────────
class Dice(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.alpha = nn.Parameter(torch.zeros(dim))
        self.bn    = nn.BatchNorm1d(dim, eps=1e-8)

    def forward(self, x):
        p = torch.sigmoid(self.bn(x))
        return p * x + (1.0 - p) * self.alpha * x


# ── DCNv2 cross layer: x_out = x0 * W(x) + x ──────────────────────────────────
class CrossLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, dim)

    def forward(self, x0, x):
        return x0 * self.linear(x) + x


# ── Full model ─────────────────────────────────────────────────────────────────
class SASRecDCNv2CTR(nn.Module):
    ITEM_VOCAB  = 91719
    ITEM_ID_DIM = 64
    MM_DIM      = 128
    SEQ_DIM     = 192        # ITEM_ID_DIM + MM_DIM
    USER_VOCAB  = 1_000_001
    USER_DIM    = 64
    LIKES_VOCAB = 11
    LIKES_DIM   = 16
    VIEWS_VOCAB = 11
    VIEWS_DIM   = 16
    MAX_LEN     = 64
    # Target attention: seq repr is 384-dim (last-pos 192 || target-attended 192)
    DCN_IN      = 672        # 384 + 192 + 64 + 16 + 16
    DEEP_OUT    = 256
    COMBINED    = 928        # DCN_IN + DEEP_OUT

    def __init__(self, item_emb_path):
        super().__init__()

        # Item ID embedding (starts frozen; unfrozen after epoch 2)
        self.item_id_emb = nn.Embedding(self.ITEM_VOCAB, self.ITEM_ID_DIM, padding_idx=0)

        # Multimodal embedding — ALWAYS frozen
        mm_np  = np.load(item_emb_path)                        # (91718, 128)
        mm_pad = np.zeros((self.ITEM_VOCAB, self.MM_DIM), dtype=np.float32)
        mm_pad[1:] = mm_np
        self.mm_emb = nn.Embedding.from_pretrained(
            torch.tensor(mm_pad), freeze=True, padding_idx=0)

        # User / popularity embeddings
        self.user_emb  = nn.Embedding(self.USER_VOCAB,  self.USER_DIM)
        self.likes_emb = nn.Embedding(self.LIKES_VOCAB, self.LIKES_DIM)
        self.views_emb = nn.Embedding(self.VIEWS_VOCAB, self.VIEWS_DIM)

        # Learnable positional encoding
        self.pos_enc = nn.Embedding(self.MAX_LEN + 1, self.SEQ_DIM)

        # Transformer encoder: 4 layers, 4 heads
        enc_layer = nn.TransformerEncoderLayer(
            d_model=self.SEQ_DIM, nhead=4, dim_feedforward=768,
            dropout=0.2, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=4)

        # Cross network (3 layers) — 672-dim input
        self.cross = nn.ModuleList([CrossLayer(self.DCN_IN) for _ in range(3)])

        # Deep network with Dice — 672-dim input
        self.deep = nn.Sequential(
            nn.Linear(self.DCN_IN, 1024), Dice(1024), nn.Dropout(0.2),
            nn.Linear(1024, 512),         Dice(512),  nn.Dropout(0.2),
            nn.Linear(512, 256),          Dice(256),  nn.Dropout(0.2),
        )

        # Prediction head: 928 → 64 → 32 → 1
        self.pred = nn.Sequential(
            nn.Linear(self.COMBINED, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

        self._init_weights()
        self.freeze_item_id_emb()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding) and m.padding_idx is not None:
                nn.init.normal_(m.weight, 0, 0.01)
                m.weight.data[m.padding_idx].zero_()

    def freeze_item_id_emb(self):
        for p in self.item_id_emb.parameters():
            p.requires_grad_(False)

    def unfreeze_item_id_emb(self):
        for p in self.item_id_emb.parameters():
            p.requires_grad_(True)

    def get_item_repr(self, ids):
        return torch.cat([self.item_id_emb(ids), self.mm_emb(ids)], dim=-1)

    def forward(self, item_seq, target_item_id, user_id, likes_level, views_level):
        B, L = item_seq.shape

        # Sequence embedding + positional encoding
        seq_repr = self.get_item_repr(item_seq)                           # (B, L, 192)
        pos      = torch.arange(1, L + 1, device=item_seq.device).unsqueeze(0)
        seq_repr = seq_repr + self.pos_enc(pos)

        # Transformer encoder with padding mask
        pad_mask = (item_seq == 0)                                        # (B, L)
        seq_out  = self.transformer(seq_repr, src_key_padding_mask=pad_mask)

        # Last non-padding position
        lengths  = (item_seq != 0).sum(dim=1).clamp(min=1)
        last_idx = (lengths - 1).clamp(min=0)
        last_out = seq_out[torch.arange(B, device=item_seq.device), last_idx]  # (B, 192)

        # Target item representation
        target_repr = self.get_item_repr(target_item_id)                  # (B, 192)

        # Target attention (DIN-style): Q=target, K=V=seq_out
        scores   = torch.bmm(target_repr.unsqueeze(1),
                             seq_out.transpose(1, 2)).squeeze(1)          # (B, L)
        scores   = scores / (self.SEQ_DIM ** 0.5)
        scores   = scores.masked_fill(pad_mask, float('-inf'))
        weights  = torch.softmax(scores, dim=1)                           # (B, L)
        attended = torch.bmm(weights.unsqueeze(1), seq_out).squeeze(1)   # (B, 192)

        # Sequence representation: last-pos || target-attended → 384-dim
        seq_representation = torch.cat([last_out, attended], dim=-1)     # (B, 384)

        # Side features
        user_e  = self.user_emb(user_id)                                  # (B,  64)
        likes_e = self.likes_emb(likes_level)                             # (B,  16)
        views_e = self.views_emb(views_level)                             # (B,  16)

        # DCNv2 — 672-dim
        x  = torch.cat([seq_representation, target_repr, user_e, likes_e, views_e], dim=-1)
        x0 = x
        xc = x
        for layer in self.cross:
            xc = layer(x0, xc)
        xd = self.deep(x)

        combined = torch.cat([xc, xd], dim=-1)                           # (B, 928)
        return torch.sigmoid(self.pred(combined)).squeeze(-1)             # (B,)


# ── Quick sanity check ─────────────────────────────────────────────────────────
_tmp      = SASRecDCNv2CTR(ITEM_EMB_PATH)
total_p   = sum(p.numel() for p in _tmp.parameters())
trainable = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f'SASRecDCNv2CTR (v3) instantiated')
print(f'  Total params     : {total_p:,}')
print(f'  Trainable params : {trainable:,}  (item_id_emb frozen at init)')
print(f'  DCN_IN / COMBINED: {_tmp.DCN_IN} / {_tmp.COMBINED}')
print(f'  Improvement      : DIN-style target attention (384-dim seq repr)')
print(f'  Loss             : BCE (stable; focal+aux caused AUC regression)')
del _tmp


SASRecDCNv2CTR (v3) instantiated
  Total params     : 86,171,393
  Trainable params : 68,561,345  (item_id_emb frozen at init)
  DCN_IN / COMBINED: 672 / 928
  Improvement      : DIN-style target attention (384-dim seq repr)
  Loss             : BCE (stable; focal+aux caused AUC regression)


/tmp/ipykernel_57/2882644462.py:72: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=4)


# Cell 4: Task 2 Training
Stratified 50% sample, linear warmup + cosine annealing LR, item_id_emb unfrozen after epoch 2.

In [8]:
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

# ── Helpers ────────────────────────────────────────────────────────────────────
def parse_seq(seq, max_len=MAX_SEQ_LEN):
    if isinstance(seq, (list, np.ndarray)):
        ids = [int(x) for x in seq if int(x) != 0]
    elif isinstance(seq, str) and seq.strip():
        ids = [int(x) for x in seq.replace(' ', '').split(',') if x and x != '0']
    else:
        ids = []
    ids = ids[-max_len:]
    return [0] * (max_len - len(ids)) + ids


class CTRDataset(Dataset):
    def __init__(self, df, seq_col, has_label=True):
        print('  Parsing sequences …')
        seqs = [parse_seq(s) for s in tqdm(df[seq_col], leave=False)]
        self.item_seq = torch.tensor(seqs,                      dtype=torch.long)
        self.item_id  = torch.tensor(df['item_id'].values,      dtype=torch.long)
        self.user_id  = torch.tensor(df['user_id'].values,      dtype=torch.long)
        self.likes    = torch.tensor(df['likes_level'].values,  dtype=torch.long)
        self.views    = torch.tensor(df['views_level'].values,  dtype=torch.long)
        self.labels   = torch.tensor(df['label'].values,        dtype=torch.float32) if has_label else None

    def __len__(self):
        return len(self.item_id)

    def __getitem__(self, idx):
        base = (self.item_seq[idx], self.item_id[idx], self.user_id[idx],
                self.likes[idx], self.views[idx])
        return base + (self.labels[idx],) if self.labels is not None else base


# ── Load & sample data ─────────────────────────────────────────────────────────
print('Loading train.parquet …')
train_raw = pd.read_parquet(f'{DATA_DIR}/train.parquet')
print(f'  shape: {train_raw.shape}  cols: {train_raw.columns.tolist()}')

seq_col = next((c for c in ['item_seq', 'item_sequence', 'his_item_id']
                if c in train_raw.columns), None)
print(f'  seq column: {seq_col}')

pos_s = train_raw[train_raw['label'] == 1].sample(frac=0.5, random_state=42)
neg_s = train_raw[train_raw['label'] == 0].sample(frac=0.5, random_state=42)
train_df = pd.concat([pos_s, neg_s]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'  sampled: {len(train_df):,}  (pos={len(pos_s):,}  neg={len(neg_s):,})')
del train_raw, pos_s, neg_s

print('Loading valid.parquet …')
valid_raw = pd.read_parquet(f'{DATA_DIR}/valid.parquet')
print(f'  shape: {valid_raw.shape}')

print('Building datasets …')
train_ds = CTRDataset(train_df,  seq_col)
valid_ds = CTRDataset(valid_raw, seq_col)
del train_df, valid_raw

# Scale batch sizes with GPU count
TRAIN_BS_T2 = 2048 * N_GPUS
VALID_BS_T2 = 4096 * N_GPUS

train_loader_t2 = DataLoader(train_ds, batch_size=TRAIN_BS_T2, shuffle=True,
                              num_workers=2, pin_memory=True)
valid_loader_t2 = DataLoader(valid_ds, batch_size=VALID_BS_T2, shuffle=False,
                              num_workers=2, pin_memory=True)

# ── Model + optimiser ──────────────────────────────────────────────────────────
_model_t2    = SASRecDCNv2CTR(ITEM_EMB_PATH).to(DEVICE)
model_t2     = nn.DataParallel(_model_t2) if N_GPUS > 1 else _model_t2
criterion    = nn.BCELoss()
optimizer_t2 = torch.optim.Adam(_model_t2.parameters(), lr=5e-4, weight_decay=1e-5)

MAX_EPOCHS_T2 = 40
WARMUP_T2     = 3
MAX_LR_T2     = 2e-4
MIN_LR_T2     = 1e-5
PATIENCE_T2   = 8

cosine_t2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_t2, T_max=MAX_EPOCHS_T2 - WARMUP_T2, eta_min=MIN_LR_T2)

best_auc_t2 = 0.0
no_imp_t2   = 0

print(f'\nTask 2 training (max {MAX_EPOCHS_T2} epochs, patience={PATIENCE_T2}, {N_GPUS} GPU(s)) …')
print(f'  Loss: BCE  |  Architecture: target attention (384-dim seq repr)')

for epoch in range(1, MAX_EPOCHS_T2 + 1):
    # ── LR schedule ──
    if epoch <= WARMUP_T2:
        lr = MAX_LR_T2 * epoch / WARMUP_T2
        for pg in optimizer_t2.param_groups:
            pg['lr'] = lr
    else:
        cosine_t2.step()
        lr = cosine_t2.get_last_lr()[0]

    # Unfreeze item_id_emb after warmup
    if epoch == WARMUP_T2 + 1:
        _model_t2.unfreeze_item_id_emb()
        print(f'  \u2191 item_id_emb unfrozen at epoch {epoch}')

    # ── Train ──
    model_t2.train()
    epoch_loss, t_ep = 0.0, time.time()

    for batch in tqdm(train_loader_t2, desc=f'Ep {epoch:02d} train', leave=False):
        seqs, iids, uids, lk, vw, labels = [b.to(DEVICE) for b in batch]
        preds = model_t2(seqs, iids, uids, lk, vw)
        loss  = criterion(preds, labels)
        optimizer_t2.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(_model_t2.parameters(), 1.0)
        optimizer_t2.step()
        epoch_loss += loss.item()

    ep_time = time.time() - t_ep
    if epoch == 1:
        print(f'  Epoch 1 in {ep_time:.1f}s — est. total: {ep_time * MAX_EPOCHS_T2 / 60:.1f} min')

    # ── Validate ──
    model_t2.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for batch in tqdm(valid_loader_t2, desc=f'Ep {epoch:02d} valid', leave=False):
            seqs, iids, uids, lk, vw, labels = [b.to(DEVICE) for b in batch]
            all_p.extend(model_t2(seqs, iids, uids, lk, vw).cpu().numpy())
            all_l.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_l, all_p)
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS_T2} | Loss {epoch_loss/len(train_loader_t2):.4f} '
          f'| AUC {auc:.4f} | LR {lr:.2e} | {ep_time:.0f}s')

    if auc > best_auc_t2:
        best_auc_t2 = auc
        no_imp_t2   = 0
        torch.save({'epoch': epoch, 'model_state': _model_t2.state_dict(), 'auc': auc}, CKPT_T2)
        print(f'  \u2713 Best saved (AUC={best_auc_t2:.4f})')
    else:
        no_imp_t2 += 1
        if no_imp_t2 >= PATIENCE_T2:
            print(f'  Early stop at epoch {epoch}')
            break

print(f'\nTask 2 best valid AUC: {best_auc_t2:.4f}')


Loading train.parquet …
  shape: (3600000, 6)  cols: ['user_id', 'item_seq', 'item_id', 'likes_level', 'views_level', 'label']
  seq column: item_seq
  sampled: 1,800,000  (pos=450,000  neg=1,350,000)
Loading valid.parquet …
  shape: (10000, 6)
Building datasets …
  Parsing sequences …


  Parsing sequences …


/tmp/ipykernel_57/2882644462.py:72: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=4)



Task 2 training (max 40 epochs, patience=8, 2 GPU(s)) …
  Loss: BCE  |  Architecture: target attention (384-dim seq repr)


  Epoch 1 in 405.1s — est. total: 270.1 min


...

Epoch 36/40 | Loss 0.0129 | AUC 0.9277 | LR 1.54e-05 | 394s


Epoch 37/40 | Loss 0.0126 | AUC 0.9289 | LR 1.31e-05 | 393s


Epoch 38/40 | Loss 0.0125 | AUC 0.9323 | LR 1.14e-05 | 394s
  ✓ Best saved (AUC=0.9323)


Epoch 39/40 | Loss 0.0123 | AUC 0.9315 | LR 1.03e-05 | 395s


Epoch 40/40 | Loss 0.0123 | AUC 0.9333 | LR 1.00e-05 | 394s
  ✓ Best saved (AUC=0.9333)

Task 2 best valid AUC: 0.9333


# Cell 5: Task 1 Eval — Frozen Multimodal Embeddings Only
Same architecture as Task 2. Only `mm_emb` (128-dim pretrained from item_emb_task1.npy) is frozen.
`item_id_emb`, `user_emb`, `likes_emb`, `views_emb`, transformer, and DCNv2 all remain trainable.

In [9]:
# ── Load data (reuse CTRDataset / parse_seq defined in Cell 4) ─────────────────
print('Loading train.parquet for Task 1 eval …')
t1e_raw = pd.read_parquet(f'{DATA_DIR}/train.parquet')
seq_col_t1e = next((c for c in ['item_seq', 'item_sequence', 'his_item_id']
                    if c in t1e_raw.columns), None)

pos_s  = t1e_raw[t1e_raw['label'] == 1].sample(frac=0.5, random_state=42)
neg_s  = t1e_raw[t1e_raw['label'] == 0].sample(frac=0.5, random_state=42)
t1e_df = pd.concat([pos_s, neg_s]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'  sampled: {len(t1e_df):,}')
del t1e_raw, pos_s, neg_s

t1e_valid = pd.read_parquet(f'{DATA_DIR}/valid.parquet')

print('Building datasets …')
t1e_train_ds = CTRDataset(t1e_df,    seq_col_t1e)
t1e_valid_ds = CTRDataset(t1e_valid, seq_col_t1e)
del t1e_df, t1e_valid

t1e_train_loader = DataLoader(t1e_train_ds, batch_size=2048 * N_GPUS, shuffle=True,
                               num_workers=2, pin_memory=True)
t1e_valid_loader = DataLoader(t1e_valid_ds, batch_size=4096 * N_GPUS, shuffle=False,
                               num_workers=2, pin_memory=True)

# ── Model: freeze ONLY multimodal embeddings (mm_emb) ─────────────────────────
_model_t1e = SASRecDCNv2CTR(ITEM_EMB_PATH).to(DEVICE)
model_t1e  = nn.DataParallel(_model_t1e) if N_GPUS > 1 else _model_t1e

# mm_emb is already frozen via freeze=True in from_pretrained — keep it frozen.
# Unfreeze item_id_emb (frozen by default in __init__) so it can learn ID signals.
_model_t1e.unfreeze_item_id_emb()
# user_emb, likes_emb, views_emb are trainable by default — leave them as-is.

trainable_t1e = sum(p.numel() for p in _model_t1e.parameters() if p.requires_grad)
frozen_t1e    = sum(p.numel() for p in _model_t1e.parameters() if not p.requires_grad)
print(f'Task 1 eval — trainable: {trainable_t1e:,}  frozen: {frozen_t1e:,}  ({N_GPUS} GPU(s))')
print(f'  Frozen  : mm_emb (128-dim pretrained multimodal embeddings only)')
print(f'  Trainable: item_id_emb, user_emb, likes_emb, views_emb + transformer/DCNv2')
print(f'  Loss: BCE')

criterion_t1e = nn.BCELoss()
optimizer_t1e = torch.optim.Adam(
    filter(lambda p: p.requires_grad, _model_t1e.parameters()),
    lr=5e-4, weight_decay=1e-5)

MAX_EPOCHS_T1E = 20
WARMUP_T1E     = 2
PATIENCE_T1E   = 5

cosine_t1e = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_t1e, T_max=MAX_EPOCHS_T1E - WARMUP_T1E, eta_min=1e-5)

best_auc_t1e = 0.0
no_imp_t1e   = 0

print(f'\nTask 1 eval training ({MAX_EPOCHS_T1E} epochs) …')

for epoch in range(1, MAX_EPOCHS_T1E + 1):
    if epoch <= WARMUP_T1E:
        lr = 5e-4 * epoch / WARMUP_T1E
        for pg in optimizer_t1e.param_groups:
            pg['lr'] = lr
    else:
        cosine_t1e.step()
        lr = cosine_t1e.get_last_lr()[0]

    model_t1e.train()
    epoch_loss, t_ep = 0.0, time.time()

    for batch in tqdm(t1e_train_loader, desc=f'T1E Ep {epoch:02d}', leave=False):
        seqs, iids, uids, lk, vw, labels = [b.to(DEVICE) for b in batch]
        preds = model_t1e(seqs, iids, uids, lk, vw)
        loss  = criterion_t1e(preds, labels)
        optimizer_t1e.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(_model_t1e.parameters(), 0.5)
        optimizer_t1e.step()
        epoch_loss += loss.item()

    ep_time = time.time() - t_ep
    if epoch == 1:
        print(f'  Epoch 1 in {ep_time:.1f}s — est. total: {ep_time * MAX_EPOCHS_T1E / 60:.1f} min')

    model_t1e.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for batch in tqdm(t1e_valid_loader, desc='T1E valid', leave=False):
            seqs, iids, uids, lk, vw, labels = [b.to(DEVICE) for b in batch]
            all_p.extend(model_t1e(seqs, iids, uids, lk, vw).cpu().numpy())
            all_l.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_l, all_p)
    print(f'T1E Ep {epoch:02d}/{MAX_EPOCHS_T1E} | Loss {epoch_loss/len(t1e_train_loader):.4f} '
          f'| AUC {auc:.4f} | LR {lr:.2e} | {ep_time:.0f}s')

    if auc > best_auc_t1e:
        best_auc_t1e = auc
        no_imp_t1e   = 0
        torch.save({'epoch': epoch, 'model_state': _model_t1e.state_dict(), 'auc': auc}, CKPT_T1E)
        print(f'  \u2713 Best saved (AUC={best_auc_t1e:.4f})')
    else:
        no_imp_t1e += 1
        if no_imp_t1e >= PATIENCE_T1E:
            print(f'  Early stop at epoch {epoch}')
            break

print(f'\nTask 1 eval best AUC: {best_auc_t1e:.4f}')


Loading train.parquet for Task 1 eval …
  sampled: 1,800,000
Building datasets …
  Parsing sequences …


  Parsing sequences …


/tmp/ipykernel_57/2882644462.py:72: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=4)


Task 1 eval — trainable: 74,431,361  frozen: 11,740,032  (2 GPU(s))
  Frozen  : mm_emb (128-dim pretrained multimodal embeddings only)
  Trainable: item_id_emb, user_emb, likes_emb, views_emb + transformer/DCNv2
  Loss: BCE

Task 1 eval training (20 epochs) …


  Epoch 1 in 397.4s — est. total: 132.5 min


...

T1E Ep 16/20 | Loss 0.0165 | AUC 0.8481 | LR 6.73e-05 | 393s
  ✓ Best saved (AUC=0.8481)


T1E Ep 17/20 | Loss 0.0158 | AUC 0.8590 | LR 4.28e-05 | 394s
  ✓ Best saved (AUC=0.8590)


T1E Ep 18/20 | Loss 0.0151 | AUC 0.8678 | LR 2.48e-05 | 393s
  ✓ Best saved (AUC=0.8678)


T1E Ep 19/20 | Loss 0.0147 | AUC 0.8638 | LR 1.37e-05 | 393s


T1E Ep 20/20 | Loss 0.0146 | AUC 0.8713 | LR 1.00e-05 | 394s
  ✓ Best saved (AUC=0.8713)

Task 1 eval best AUC: 0.8713


# Cell 6: Inference — Task 1
Load `task1_eval_best.pt`, score `test.parquet`, save `sub_task1_v1.csv`.

In [10]:
from torch.utils.data import Dataset, DataLoader

class TestDataset(Dataset):
    def __init__(self, df, seq_col):
        print('  Parsing test sequences …')
        seqs = [parse_seq(s) for s in tqdm(df[seq_col], leave=False)]
        self.item_seq = torch.tensor(seqs,                     dtype=torch.long)
        self.item_id  = torch.tensor(df['item_id'].values,     dtype=torch.long)
        self.user_id  = torch.tensor(df['user_id'].values,     dtype=torch.long)
        self.likes    = torch.tensor(df['likes_level'].values,  dtype=torch.long)
        self.views    = torch.tensor(df['views_level'].values,  dtype=torch.long)

    def __len__(self):  return len(self.item_id)

    def __getitem__(self, idx):
        return (self.item_seq[idx], self.item_id[idx], self.user_id[idx],
                self.likes[idx], self.views[idx])


print('Loading test.parquet …')
test_df  = pd.read_parquet(f'{DATA_DIR}/test.parquet')
print(f'  shape: {test_df.shape}  cols: {test_df.columns.tolist()}')

id_col       = next((c for c in ['ID', 'id', 'sample_id'] if c in test_df.columns),
                    test_df.columns[0])
seq_col_test = next((c for c in ['item_seq', 'item_sequence', 'his_item_id']
                     if c in test_df.columns), None)
print(f'  ID col: {id_col}  seq col: {seq_col_test}')

test_ds     = TestDataset(test_df, seq_col_test)
test_loader = DataLoader(test_ds, batch_size=4096 * N_GPUS, shuffle=False,
                         num_workers=2, pin_memory=True)

# Load checkpoint into plain model, then wrap with DataParallel
ckpt_t1e_inf  = torch.load(CKPT_T1E, map_location=DEVICE, weights_only=False)
_inf_t1e      = SASRecDCNv2CTR(ITEM_EMB_PATH).to(DEVICE)
_inf_t1e.load_state_dict(ckpt_t1e_inf['model_state'])
inf_model_t1e = nn.DataParallel(_inf_t1e) if N_GPUS > 1 else _inf_t1e
inf_model_t1e.eval()
print(f'Loaded task1_eval_best.pt  (epoch={ckpt_t1e_inf["epoch"]},  AUC={ckpt_t1e_inf["auc"]:.4f})')

scores_t1 = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Task1 inference'):
        seqs, iids, uids, lk, vw = [b.to(DEVICE) for b in batch]
        scores_t1.extend(inf_model_t1e(seqs, iids, uids, lk, vw).cpu().numpy().tolist())

assert len(scores_t1) == 379142, f'Row count mismatch: {len(scores_t1)}'
print(f'Row count verified: {len(scores_t1):,} \u2713')

sub_t1      = pd.DataFrame({'ID': test_df[id_col].values, 'label': scores_t1})
sub_t1_path = f'{OUTPUT_DIR}/submissions/sub_task1_v1.csv'
sub_t1.to_csv(sub_t1_path, index=False)
print(f'Saved: {sub_t1_path}')
print(sub_t1.head())


Loading test.parquet …
  shape: (379142, 6)  cols: ['ID', 'user_id', 'item_seq', 'item_id', 'likes_level', 'views_level']
  ID col: ID  seq col: item_seq
  Parsing test sequences …


/tmp/ipykernel_57/2882644462.py:72: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=4)


Loaded task1_eval_best.pt  (epoch=20,  AUC=0.8713)


Task1 inference: 100%|██████████| 47/47 [00:27<00:00,  1.70it/s]


Row count verified: 379,142 ✓
Saved: /kaggle/working/outputs/submissions/sub_task1_v1.csv
   ID         label
0   0  1.830298e-03
1   1  1.000000e+00
2   2  8.253898e-01
3   3  8.087205e-36
4   4  0.000000e+00


# Cell 7: Inference — Task 2
Load `task2_best.pt`, score `test.parquet`, save `sub_task2_v1.csv`.

In [11]:
ckpt_t2_inf  = torch.load(CKPT_T2, map_location=DEVICE,  weights_only=False)
_inf_t2      = SASRecDCNv2CTR(ITEM_EMB_PATH).to(DEVICE)
_inf_t2.load_state_dict(ckpt_t2_inf['model_state'])
inf_model_t2 = nn.DataParallel(_inf_t2) if N_GPUS > 1 else _inf_t2
inf_model_t2.eval()
print(f'Loaded task2_best.pt  (epoch={ckpt_t2_inf["epoch"]},  AUC={ckpt_t2_inf["auc"]:.4f})')

scores_t2 = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Task2 inference'):
        seqs, iids, uids, lk, vw = [b.to(DEVICE) for b in batch]
        scores_t2.extend(inf_model_t2(seqs, iids, uids, lk, vw).cpu().numpy().tolist())

assert len(scores_t2) == 379142, f'Row count mismatch: {len(scores_t2)}'
print(f'Row count verified: {len(scores_t2):,} \u2713')

sub_t2      = pd.DataFrame({'ID': test_df[id_col].values, 'label': scores_t2})
sub_t2_path = f'{OUTPUT_DIR}/submissions/sub_task2_v1.csv'
sub_t2.to_csv(sub_t2_path, index=False)
print(f'Saved: {sub_t2_path}')
print(sub_t2.head())


/tmp/ipykernel_57/2882644462.py:72: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=4)


Loaded task2_best.pt  (epoch=40,  AUC=0.9333)


Task2 inference: 100%|██████████| 47/47 [00:27<00:00,  1.72it/s]


Row count verified: 379,142 ✓
Saved: /kaggle/working/outputs/submissions/sub_task2_v1.csv
   ID         label
0   0  9.541127e-22
1   1  1.000000e+00
2   2  1.022751e-13
3   3  0.000000e+00
4   4  0.000000e+00


# Cell 8: Final Summary
Print AUCs, verify row counts, copy CSVs to `/kaggle/working/` for download.

In [17]:
import pandas as pd, zipfile

# Load both CSVs
sub_t1 = pd.read_csv('/kaggle/working/outputs/submissions/sub_task1_v1.csv')  # columns: ID, label
sub_t2 = pd.read_csv('/kaggle/working/outputs/submissions/sub_task2_v1.csv')  # columns: ID, label

# Merge on ID and rename columns
merged = sub_t1.rename(columns={'label': 'Task1'}).merge(
    sub_t2.rename(columns={'label': 'Task2'}), on='ID'
)

# Task1&2 = average of Task1 and Task2 scores
merged['Task1&2'] = (merged['Task1'] + merged['Task2']) / 2

# Reorder columns exactly as required
merged = merged[['ID', 'Task1', 'Task2', 'Task1&2']]

# Verify
assert len(merged) == 379142, f'Row count wrong: {len(merged)}'
assert list(merged.columns) == ['ID', 'Task1', 'Task2', 'Task1&2']
print(f'Rows    : {len(merged):,} \u2713')
print(f'Columns : {list(merged.columns)} \u2713')
print(merged.head())
print(f'Task1   range: [{merged.Task1.min():.4f}, {merged.Task1.max():.4f}]')
print(f'Task2   range: [{merged.Task2.min():.4f}, {merged.Task2.max():.4f}]')
print(f'Task1&2 range: [{merged["Task1&2"].min():.4f}, {merged["Task1&2"].max():.4f}]')

# Save with exact filename required by competition (typo is intentional)
merged.to_csv('/kaggle/working/prediction.csv', index=False)
print('Saved: prediction.csv')

# Zip for submission
with zipfile.ZipFile('/kaggle/working/submission.zip', 'w',
                     compression=zipfile.ZIP_DEFLATED) as z:
    z.write('/kaggle/working/prediction.csv', 'predictioin.csv')
print('Created: prediction.zip — download this and upload to Codabench')


Rows    : 379,142 ✓
Columns : ['ID', 'Task1', 'Task2', 'Task1&2'] ✓
   ID         Task1         Task2       Task1&2
0   0  1.830298e-03  9.541127e-22  9.151490e-04
1   1  1.000000e+00  1.000000e+00  1.000000e+00
2   2  8.253898e-01  1.022751e-13  4.126949e-01
3   3  8.087205e-36  0.000000e+00  4.043602e-36
4   4  0.000000e+00  0.000000e+00  0.000000e+00
Task1   range: [0.0000, 1.0000]
Task2   range: [0.0000, 1.0000]
Task1&2 range: [0.0000, 1.0000]
Saved: predictioin.csv
Created: prediction.zip — download this and upload to Codabench


In [18]:
import shutil

SEP = '=' * 60
print(SEP)
print('FINAL SUMMARY')
print(SEP)
print(f'Task 1 eval best valid AUC : {best_auc_t1e:.4f}')
print(f'Task 2     best valid AUC  : {best_auc_t2:.4f}')
print()

for path in [sub_t1_path, sub_t2_path]:
    df = pd.read_csv(path)
    print(f'{path}')
    print(f'  rows : {len(df):,}')
    print(f'  label: [{df["label"].min():.6f}, {df["label"].max():.6f}]')

print()
print('Copying submissions to /kaggle/working/ …')
for src, dst in [
    (sub_t1_path, '/kaggle/working/sub_task1_v1.csv'),
    (sub_t2_path, '/kaggle/working/sub_task2_v1.csv'),
]:
    shutil.copy(src, dst)
    print(f'  {src} -> {dst}')

print()
print('All output files:')
for root, dirs, files in os.walk(OUTPUT_DIR):
    for fname in sorted(files):
        fp      = os.path.join(root, fname)
        size_mb = os.path.getsize(fp) / 1e6
        print(f'  {fp}  ({size_mb:.1f} MB)')

print()
print(SEP)
print('DONE \u2713')
print(SEP)

FINAL SUMMARY
Task 1 eval best valid AUC : 0.8713
Task 2     best valid AUC  : 0.9333

/kaggle/working/outputs/submissions/sub_task1_v1.csv
  rows : 379,142
  label: [0.000000, 1.000000]
/kaggle/working/outputs/submissions/sub_task2_v1.csv
  rows : 379,142
  label: [0.000000, 1.000000]

Copying submissions to /kaggle/working/ …
  /kaggle/working/outputs/submissions/sub_task1_v1.csv -> /kaggle/working/sub_task1_v1.csv
  /kaggle/working/outputs/submissions/sub_task2_v1.csv -> /kaggle/working/sub_task2_v1.csv

All output files:
  /kaggle/working/outputs/features/item_emb_task1.npy  (47.0 MB)
  /kaggle/working/outputs/checkpoints/task1_eval_best.pt  (344.7 MB)
  /kaggle/working/outputs/checkpoints/task2_best.pt  (344.7 MB)
  /kaggle/working/outputs/submissions/sub_task1_v1.csv  (8.7 MB)
  /kaggle/working/outputs/submissions/sub_task2_v1.csv  (8.7 MB)

DONE ✓
